# MobileNetV3 의류 분류 학습 Colab 노트북

이 노트북은 **GitHub clone 없이**, Google Drive에 있는 데이터셋만 사용해서 MobileNetV3-Small 의류 분류 모델을 학습합니다.

예상 데이터셋 구조:

```text
/content/drive/MyDrive/fitcommit_clothing_dataset/
  train/
    T-shirt/
    Shirt/
    Hoodie/
    Sweater/
    Pants/
    Skirt/
    Dress/
    Outerwear/
  val/
    T-shirt/
    Shirt/
    Hoodie/
    Sweater/
    Pants/
    Skirt/
    Dress/
    Outerwear/
  test/   # 선택 사항
    ...
```

학습 결과는 Google Drive에 저장됩니다.

```text
/content/drive/MyDrive/fitcommit_models/clothing_mobilenet_v3_small.pth
```


## 1. GPU 확인

`cuda available: False`가 나오면 Colab에서 `Runtime → Change runtime type → T4 GPU`로 바꾼 뒤 다시 실행하세요.


In [1]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("GPU가 연결되지 않았습니다. Runtime을 T4 GPU로 바꿔주세요.")

torch: 2.11.0+cu128
cuda available: True
gpu: Tesla T4


## 2. Google Drive 연결

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 3. 경로와 클래스 설정

Drive에 데이터셋 폴더명이 다르면 `DATA_DIR`만 수정하세요.


In [3]:
!pip install -q kagglehub

In [4]:
import kagglehub

raw_path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")
print("Path to dataset files:", raw_path)

Using Colab cache for faster access to the 'fashion-product-images-small' dataset.
Path to dataset files: /kaggle/input/fashion-product-images-small


In [5]:
from pathlib import Path

raw_path = Path(raw_path)
print(raw_path)
print(list(raw_path.iterdir())[:10])

/kaggle/input/fashion-product-images-small
[PosixPath('/kaggle/input/fashion-product-images-small/myntradataset'), PosixPath('/kaggle/input/fashion-product-images-small/images'), PosixPath('/kaggle/input/fashion-product-images-small/styles.csv')]


In [7]:
import pandas as pd
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

RAW_DIR = Path(raw_path)
IMAGES_DIR = RAW_DIR / "images"
CSV_PATH = RAW_DIR / "styles.csv"

OUTPUT_DIR = Path("/content/drive/MyDrive/fitcommit_clothing_dataset")

df = pd.read_csv(CSV_PATH, on_bad_lines="skip")

print(df.columns)
print(df[["id", "masterCategory", "subCategory", "articleType"]].head())

Index(['id', 'gender', 'masterCategory', 'subCategory', 'articleType',
       'baseColour', 'season', 'year', 'usage', 'productDisplayName'],
      dtype='object')
      id masterCategory subCategory  articleType
0  15970        Apparel     Topwear       Shirts
1  39386        Apparel  Bottomwear        Jeans
2  59263    Accessories     Watches      Watches
3  21379        Apparel  Bottomwear  Track Pants
4  53759        Apparel     Topwear      Tshirts


In [8]:
import random

CATEGORIES = [
    "T-shirt",
    "Shirt",
    "Hoodie",
    "Sweater",
    "Pants",
    "Skirt",
    "Dress",
    "Outerwear",
]

ARTICLE_TO_CATEGORY = {
    # T-shirt
    "Tshirts": "T-shirt",
    "Tops": "T-shirt",

    # Shirt
    "Shirts": "Shirt",

    # Hoodie / Sweater
    "Sweatshirts": "Sweater",
    "Sweaters": "Sweater",

    # Pants
    "Jeans": "Pants",
    "Trousers": "Pants",
    "Track Pants": "Pants",
    "Capris": "Pants",
    "Leggings": "Pants",

    # Skirt
    "Skirts": "Skirt",

    # Dress
    "Dresses": "Dress",

    # Outerwear
    "Jackets": "Outerwear",
    "Blazers": "Outerwear",
    "Shrug": "Outerwear",
    "Coats": "Outerwear",
}

df["category"] = df["articleType"].map(ARTICLE_TO_CATEGORY)
df = df.dropna(subset=["category"]).copy()

df["image_path"] = df["id"].astype(str).apply(lambda x: IMAGES_DIR / f"{x}.jpg")
df = df[df["image_path"].apply(lambda p: p.exists())].copy()

print(df["category"].value_counts())

category
T-shirt      8828
Shirt        3215
Pants        1794
Sweater       562
Dress         464
Outerwear     272
Skirt         128
Name: count, dtype: int64


In [9]:
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split in ["train", "val", "test"]:
    for category in CATEGORIES:
        (OUTPUT_DIR / split / category).mkdir(parents=True, exist_ok=True)

for category in CATEGORIES:
    category_df = df[df["category"] == category].copy()

    if len(category_df) < 10:
        print("데이터가 너무 적음:", category, len(category_df))
        continue

    train_df, temp_df = train_test_split(
        category_df,
        test_size=0.3,
        random_state=42,
        shuffle=True,
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=42,
        shuffle=True,
    )

    split_map = {
        "train": train_df,
        "val": val_df,
        "test": test_df,
    }

    for split, split_df in split_map.items():
        for _, row in split_df.iterrows():
            src = row["image_path"]
            dst = OUTPUT_DIR / split / category / src.name
            shutil.copy2(src, dst)

print("완료:", OUTPUT_DIR)

데이터가 너무 적음: Hoodie 0
완료: /content/drive/MyDrive/fitcommit_clothing_dataset


In [13]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/fitcommit_clothing_dataset")
MODEL_DIR = Path("/content/drive/MyDrive/fitcommit_models")
MODEL_PATH = MODEL_DIR / "clothing_mobilenet_v3_small.pth"

CATEGORIES = [
    "T-shirt",
    "Shirt",
    "Sweater",
    "Pants",
    "Skirt",
    "Dress",
    "Outerwear",
]

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("MODEL_PATH:", MODEL_PATH)

DATA_DIR: /content/drive/MyDrive/fitcommit_clothing_dataset
MODEL_PATH: /content/drive/MyDrive/fitcommit_models/clothing_mobilenet_v3_small.pth


## 4. 데이터셋 구조 확인

`missing` 또는 `extra`가 나오면 Drive의 폴더 이름을 수정하세요.


In [16]:
def count_images(folder: Path) -> int:
    extensions = ["*.jpg", "*.jpeg", "*.png", "*.webp", "*.bmp"]
    total = 0
    for ext in extensions:
        total += len(list(folder.glob(ext)))
        total += len(list(folder.glob(ext.upper())))
    return total

print("DATA_DIR exists:", DATA_DIR.exists())

for split in ["train", "val", "test"]:
    split_dir = DATA_DIR / split
    print(f"\n== {split} ==")
    print("exists:", split_dir.exists())

    if not split_dir.exists():
        continue

    found = [p.name for p in split_dir.iterdir() if p.is_dir()]
    print("found classes:", found)

    missing = set(CATEGORIES) - set(found)
    extra = set(found) - set(CATEGORIES)

    if missing:
        print("missing:", sorted(missing))
    if extra:
        print("extra:", sorted(extra))

    for class_name in CATEGORIES:
        class_dir = split_dir / class_name
        if class_dir.exists():
            print(f"{class_name}: {count_images(class_dir)} images")

DATA_DIR exists: True

== train ==
exists: True
found classes: ['T-shirt', 'Shirt', 'Sweater', 'Pants', 'Skirt', 'Dress', 'Outerwear']
T-shirt: 6179 images
Shirt: 2250 images
Sweater: 393 images
Pants: 1255 images
Skirt: 89 images
Dress: 324 images
Outerwear: 190 images

== val ==
exists: True
found classes: ['T-shirt', 'Shirt', 'Sweater', 'Pants', 'Skirt', 'Dress', 'Outerwear']
T-shirt: 1324 images
Shirt: 482 images
Sweater: 84 images
Pants: 269 images
Skirt: 19 images
Dress: 70 images
Outerwear: 41 images

== test ==
exists: True
found classes: ['T-shirt', 'Shirt', 'Sweater', 'Pants', 'Skirt', 'Dress', 'Outerwear']
T-shirt: 1325 images
Shirt: 483 images
Sweater: 85 images
Pants: 270 images
Skirt: 20 images
Dress: 70 images
Outerwear: 41 images


## 5. 필요한 라이브러리 import

In [3]:
NUM_CLASSES = len(CATEGORIES)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(DATA_DIR / "train", transform=train_transform)
val_dataset = datasets.ImageFolder(DATA_DIR / "val", transform=eval_transform)
test_dataset = datasets.ImageFolder(DATA_DIR / "test", transform=eval_transform)

print("classes:", train_dataset.classes)
print("num classes:", len(train_dataset.classes))
print("train:", len(train_dataset))
print("val:", len(val_dataset))
print("test:", len(test_dataset))

NameError: name 'CATEGORIES' is not defined

In [18]:
targets = [label for _, label in train_dataset.samples]
class_counts = np.bincount(targets, minlength=NUM_CLASSES)

class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

NameError: name 'train_dataset' is not defined

In [17]:
import copy
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


: 

## 6. Dataset / DataLoader 생성

주의: `ImageFolder`는 기본적으로 폴더명을 알파벳순으로 class index에 매핑합니다.  
이 노트북은 `target_transform`을 사용해 서버의 `CATEGORIES` 순서와 강제로 맞춥니다.


In [ ]:
weights = MobileNet_V3_Small_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.70, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# 먼저 ImageFolder의 원래 class 순서를 확인합니다.
_tmp_train = datasets.ImageFolder(DATA_DIR / "train")
imagefolder_classes = _tmp_train.classes
print("ImageFolder classes:", imagefolder_classes)
print("Service categories:", CATEGORIES)

# ImageFolder label index -> service label index로 변환
mapping = {imagefolder_classes.index(name): CATEGORIES.index(name) for name in CATEGORIES}
print("index mapping:", mapping)

def map_target(y):
    return mapping[y]

train_dataset = datasets.ImageFolder(
    DATA_DIR / "train",
    transform=train_transform,
    target_transform=map_target,
)

val_dataset = datasets.ImageFolder(
    DATA_DIR / "val",
    transform=val_transform,
    target_transform=map_target,
)

test_dataset = None
if (DATA_DIR / "test").exists():
    _tmp_test = datasets.ImageFolder(DATA_DIR / "test")
    if set(_tmp_test.classes) == set(CATEGORIES):
        test_dataset = datasets.ImageFolder(
            DATA_DIR / "test",
            transform=val_transform,
            target_transform=map_target,
        )

BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

test_loader = None
if test_dataset is not None:
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print("train images:", len(train_dataset))
print("val images:", len(val_dataset))
print("test images:", len(test_dataset) if test_dataset else 0)

## 7. 모델 생성

In [ ]:
model = mobilenet_v3_small(weights=weights)

in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, len(CATEGORIES))

model = model.to(device)

print(model.classifier)

## 8. 학습 함수 정의

In [ ]:
def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * images.size(0)
        total_correct += (preds == labels).sum().item()
        total_count += images.size(0)

    return total_loss / total_count, total_correct / total_count


def predict_all(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            preds = logits.argmax(dim=1).cpu().numpy().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy().tolist())

    return all_labels, all_preds

## 9. 1단계: classifier head만 학습

먼저 feature extractor는 고정하고 마지막 classifier만 학습합니다.


In [ ]:
EPOCHS_HEAD = 5
LR_HEAD = 1e-3

for param in model.features.parameters():
    param.requires_grad = False

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LR_HEAD, weight_decay=1e-4)

best_acc = 0.0
best_state = copy.deepcopy(model.state_dict())

for epoch in range(1, EPOCHS_HEAD + 1):
    train_loss, train_acc = run_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_one_epoch(model, val_loader, criterion)

    print(
        f"[HEAD {epoch:02d}/{EPOCHS_HEAD}] "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

    if val_acc > best_acc:
        best_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        print("  -> best updated")

## 10. 2단계: 전체 fine-tuning

전체 layer를 풀고 낮은 learning rate로 추가 학습합니다.


In [ ]:
EPOCHS_FINE = 15
LR_FINE = 1e-4

for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FINE)

for epoch in range(1, EPOCHS_FINE + 1):
    train_loss, train_acc = run_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_one_epoch(model, val_loader, criterion)
    scheduler.step()

    print(
        f"[FINE {epoch:02d}/{EPOCHS_FINE}] "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

    if val_acc > best_acc:
        best_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        print("  -> best updated")

model.load_state_dict(best_state)
print("best val acc:", best_acc)

## 11. Validation / Test 결과 확인

In [ ]:
val_labels, val_preds = predict_all(model, val_loader)

print("Validation classification report:")
print(classification_report(val_labels, val_preds, target_names=CATEGORIES, digits=4))

print("Validation confusion matrix:")
print(confusion_matrix(val_labels, val_preds))

test_acc = None
if test_loader is not None:
    test_labels, test_preds = predict_all(model, test_loader)
    test_acc = float(np.mean(np.array(test_labels) == np.array(test_preds)))

    print("\nTest classification report:")
    print(classification_report(test_labels, test_preds, target_names=CATEGORIES, digits=4))

    print("Test confusion matrix:")
    print(confusion_matrix(test_labels, test_preds))
else:
    print("\nNo test folder found. test_acc will be None.")

## 12. 모델 저장

FastAPI 서비스에서 label 순서가 틀리지 않도록 `categories`를 checkpoint 안에 같이 저장합니다.


In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "categories": CATEGORIES,
    "val_acc": float(best_acc),
    "test_acc": test_acc,
    "architecture": "mobilenet_v3_small",
    "input_size": 224,
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
    },
}

torch.save(checkpoint, MODEL_PATH)

print("saved:", MODEL_PATH)
print("size MB:", MODEL_PATH.stat().st_size / 1024 / 1024)

## 13. 저장된 checkpoint 확인

In [ ]:
loaded = torch.load(MODEL_PATH, map_location="cpu")

print("keys:", loaded.keys())
print("categories:", loaded["categories"])
print("val_acc:", loaded["val_acc"])
print("test_acc:", loaded["test_acc"])
print("architecture:", loaded["architecture"])

## 14. 로컬 프로젝트에 적용하기

학습이 끝난 뒤 Google Drive에서 아래 파일을 다운로드하세요.

```text
MyDrive/fitcommit_models/clothing_mobilenet_v3_small.pth
```

그리고 Mac 프로젝트의 아래 위치에 넣으면 됩니다.

```text
/Users/chan/SW-Development/SYU/Fitcommit_Development/AI/models/clothing_mobilenet_v3_small.pth
```

그 후 FastAPI를 실행하기 전에 로컬에서 모델 로드 테스트를 하면 됩니다.
